# Landmark Image Classification – Inference App

This notebook implements a simple image classification app using a model trained to recognize 50 landmark classes. The app allows users to upload a photo and see the top 5 most likely predictions returned by the model.

The interface is built using `ipywidgets`, and the model is loaded from a TorchScript export for lightweight, dependency-free inference.


## App Interface

The following code builds a basic interface using `ipywidgets`. Users can upload an image and view the top 5 predictions along with confidence scores.


In [ ]:
from ipywidgets import VBox, Button, FileUpload, Output, Label
from PIL import Image
from IPython.display import display
import io
import numpy as np
import torchvision
import torchvision.transforms as T
import torch

# Decide which model you want to use among the ones exported
learn_inf = torch.jit.load("checkpoints/transfer_exported.pt")

from src.data import get_data_loaders
learn_inf.class_names = get_data_loaders(batch_size=1)["train"].dataset.classes


def on_click_classify(change):

    # Load image that has been uploaded
    fn = io.BytesIO(btn_upload.data[-1])

    img = Image.open(fn)
    img.load()

    # Let's clear the previous output (if any)
    out_pl.clear_output()

    # Display the image
    with out_pl:

        ratio = img.size[0] / img.size[1]
        c = img.copy()
        c.thumbnail([ratio * 200, 200])
        display(c)

    # Transform to tensor
    timg = T.ToTensor()(img).unsqueeze_(0)

    # Calling the model
    softmax = learn_inf(timg).data.cpu().numpy().squeeze()

    # Get the indexes of the classes ordered by softmax
    # (larger first)
    idxs = np.argsort(softmax)[::-1]

    # Loop over the classes with the largest softmax
    for i in range(5):
        # Get softmax value
        p = softmax[idxs[i]]

        # Get class name
        landmark_name = learn_inf.class_names[idxs[i]]

        labels[i].value = f"{landmark_name} (prob: {p:.2f})"


# Putting back btn_upload to a widget for next cell
btn_upload = FileUpload()

btn_run = Button(description="Classify")
btn_run.on_click(on_click_classify)

labels = []
for _ in range(5):
    labels.append(Label())

out_pl = Output()
out_pl.clear_output()

wgs = [Label("Please upload a picture of a landmark"), btn_upload, btn_run, out_pl]
wgs.extend(labels)

VBox(wgs)

---

## Summary

This notebook provides a minimal working example of how to load a trained CNN model and build a simple inference interface using Python and ipywidgets. The model is portable and performs real-time predictions on uploaded landmark images.
